In [ ]:
from pyspark import SparkConf, SparkContext

# Step 1: Define Spark configuration
conf = SparkConf().setAppName("Spark Example").setMaster("local[*]")

# Step 2: Initialize SparkContext
sc = SparkContext(conf=conf)

# Print SparkContext information to verify setup
print(f"App Name: {sc.appName}")
print(f"Master URL: {sc.master}")

App Name: Spark Example
Master URL: local[*]


In [ ]:
# Stop the SparkContext to release resources
sc.stop()

MAX TEMP

In [ ]:
# Step 1: Set up Spark configuration and context
conf = SparkConf().setMaster("local").setAppName("MaxTemperatures")
sc = SparkContext(conf=conf)

# Step 2: Define a function to parse each line of the CSV file
def parseLine(line):
    # Split the line by commas
    fields = line.split(',')
    # Extract the station ID
    stationID = fields[0]
    # Extract the entry type (e.g., "TMIN", "TMAX")
    entryType = fields[2]
    # Convert the temperature to Fahrenheit (original data is in tenths of degrees Celsius)
    temperature = float(fields[3]) * 0.1 * (9.0 / 5.0) + 32.0
    # Return a tuple (stationID, entryType, temperature)
    return (stationID, entryType, temperature)

# Step 3: Load the temperature data CSV file as an RDD
lines = sc.textFile("2-temperature-data.csv")

# Step 4: Parse each line using the parseLine function
parsedLines = lines.map(parseLine)

# Step 5: Filter the parsed lines to only include entries marked with "TMAX" (maximum temperature)
maxTemps = parsedLines.filter(lambda x: "TMAX" in x[1])

# Step 6: Extract a key-value pair of (stationID, temperature) from each filtered entry
stationTemps = maxTemps.map(lambda x: (x[0], x[2]))

# Step 7: Reduce by key (station ID) to find the maximum temperature for each station
maxTemps = stationTemps.reduceByKey(lambda x, y: max(x, y))

# Step 8: Collect the results from the RDD to the driver
results = maxTemps.collect()

# Step 9: Print the results
# For each station, print the station ID and its maximum temperature in Fahrenheit, formatted to 2 decimal places
for result in results:
    print(result[0] + "\t{:.2f}F".format(result[1]))

# Output example:
# USC0010XXXX	104.56F
# USC0020YYYY	98.34F
sc.stop()

ITE00100554	90.14F
EZE00100082	90.14F


------------------------
# Friends by Age

**Note:**  Please remember to upload your file each time you want to run this:

In [ ]:
# Getting Data from Github
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-friends-data.csv
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-temperature-data.csv
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-book.txt
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-adverse-events.csv


# Verify the files are downloaded
!ls -lh 2-friends-data.csv 2-temperature-data.csv 2-book.txt 2-adverse-events.csv

-rw-r--r-- 1 root root 588K Feb  2 22:26 2-adverse-events.csv
-rw-r--r-- 1 root root 259K Feb  2 22:26 2-book.txt
-rw-r--r-- 1 root root 8.6K Feb  2 22:26 2-friends-data.csv
-rw-r--r-- 1 root root  62K Feb  2 22:26 2-temperature-data.csv


In [ ]:
# configure Spark
conf = SparkConf().setMaster("local").setAppName("FriendsByAge")
sc = SparkContext(conf = conf)

In [ ]:
# Step 2: Define a function to parse each line of the CSV file
def parseLine(line):
    # Split the line by commas
    fields = line.split(',')
    # Extract age and number of friends from the fields
    age = int(fields[2])
    numFriends = int(fields[3])
    # Return a tuple (age, numFriends)
    return (age, numFriends)

# Step 3: Load the CSV file as an RDD
lines = sc.textFile("2-friends-data.csv")

# Step 4: Parse each line using the parseLine function
rdd = lines.map(parseLine)

# Step 5: Prepare data for aggregation by mapping each value to a tuple (numFriends, 1)
# Then reduce by key (age) to calculate total friends and total count per age
totalsByAge = rdd.mapValues(lambda x: (x, 1)) \
                 .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# Step 6: Calculate the average number of friends per age
averagesByAge = totalsByAge.mapValues(lambda x: x[0] / x[1])

# Step 7: Collect the results and print them
results = averagesByAge.collect()
for result in results:
    print(result)

# Output example:
# (18, 35.0)
# (19, 25.5)
# (20, 40.3)


#stop the spark
sc.stop()


(33, 325.3333333333333)
(26, 242.05882352941177)
(55, 295.53846153846155)
(40, 250.8235294117647)
(68, 269.6)
(59, 220.0)
(37, 249.33333333333334)
(54, 278.0769230769231)
(38, 193.53333333333333)
(27, 228.125)
(53, 222.85714285714286)
(57, 258.8333333333333)
(56, 306.6666666666667)
(43, 230.57142857142858)
(36, 246.6)
(22, 206.42857142857142)
(35, 211.625)
(45, 309.53846153846155)
(60, 202.71428571428572)
(67, 214.625)
(19, 213.27272727272728)
(30, 235.8181818181818)
(51, 302.14285714285717)
(25, 197.45454545454547)
(21, 350.875)
(42, 303.5)
(49, 184.66666666666666)
(48, 281.4)
(50, 254.6)
(39, 169.28571428571428)
(32, 207.9090909090909)
(58, 116.54545454545455)
(64, 281.3333333333333)
(31, 267.25)
(52, 340.6363636363636)
(24, 233.8)
(20, 165.0)
(62, 220.76923076923077)
(41, 268.55555555555554)
(44, 282.1666666666667)
(69, 235.2)
(65, 298.2)
(61, 256.22222222222223)
(28, 209.1)
(66, 276.44444444444446)
(46, 223.69230769230768)
(29, 215.91666666666666)
(18, 343.375)
(47, 233.22222222222